# Cross-dimension classification: ability ↔ benevolence

Does manipulating one dimension change how the other one is expressed?

1. **Ability barycenters**: one FGW barycenter per ability level (Low, Medium, High), each from $m$ transcripts where only ability was manipulated (`full_ability_*`).
2. They classify the ability of transcripts where only **benevolence** was manipulated (`full_benevolence_*`). If the two dimensions were independent, the predicted ability would not depend on the instructed benevolence: we measure the Spearman correlation between them.
3. The same in the other direction: benevolence barycenters classify ability-manipulated transcripts.

As a reference, the same barycenters also classify held-out transcripts of **their own** dimension: the correlation with the instructed level of that dimension shows how strongly a manipulated dimension is picked up. Transcripts used for the barycenters are never classified.

To reduce the dependence on the $m$ examples, the barycenters are drawn `N_REPEATS` times; for each transcript, the distances to the barycenters of a level are averaged over repeats. Besides the predicted level, a continuous **score** $d(\text{Low}) - d(\text{High})$ (positive: closer to High) is correlated too.

In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import LEVELS, load_transcripts, spearman_with_ci
from graph_multimodalsentence import Graph, compute_barycenter
from ot_distances import Fused_Gromov_Wasserstein_distance

## Parameters

In [ ]:
DATA_DIR = "../data_with_text_tag"
OUTPUT_DIR = "results"
DIMENSIONS = ["ability", "benevolence"]

M = 10                          # transcripts per level for each barycenter
N_REPEATS = 5                   # independent draws of the M examples
EXAMPLE_POOL_SHARE = 0.2        # share of each manipulated set reserved for drawing examples
N_TARGET_PER_LEVEL = 1000       # classified transcripts per instructed level and set (None: all)
SEED = 0

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
ALPHA = 0.14                    # set to the value selected in tune_alpha.ipynb
BARYCENTER_MAX_ITER = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Transcripts

For each dimension, the transcripts where **only** that dimension was manipulated. Each set is split once into an example pool (barycenters) and targets (classified), stratified by instructed level.

In [ ]:
df = load_transcripts(DATA_DIR, DIMENSIONS, reference="instructed")
rng = np.random.default_rng(SEED)

pools, targets = {}, {}
for dimension in DIMENSIONS:
    manipulated = df[(df["dimension"] == dimension) & df["condition"].str.startswith(f"full_{dimension}_gender")]
    pool, target = [], []
    for level, group in manipulated.groupby("level"):
        group = group.sample(frac=1, random_state=SEED)
        n_pool = int(round(EXAMPLE_POOL_SHARE * len(group)))
        pool.append(group.iloc[:n_pool])
        rest = group.iloc[n_pool:]
        target.append(rest if N_TARGET_PER_LEVEL is None else rest.iloc[:N_TARGET_PER_LEVEL])
    pools[dimension] = pd.concat(pool)
    targets[dimension] = pd.concat(target)

pd.DataFrame({f"{d} {kind}": s[d].groupby("level").size() for d in DIMENSIONS
              for kind, s in [("examples pool", pools), ("targets", targets)]}).loc[LEVELS]

## Graphs

Built when first needed (only the drawn examples and the targets).

In [ ]:
class CachedEncoder:
    # Wraps a SentenceTransformer so that each description is encoded only once.
    def __init__(self, model):
        self.model = model
        self.cache = {}

    def encode(self, text):
        if text not in self.cache:
            self.cache[text] = self.model.encode(text)
        return self.cache[text]


embedding = CachedEncoder(SentenceTransformer(EMBEDDING_MODEL))
texts = pd.concat([*pools.values(), *targets.values()]).drop_duplicates("id").set_index("id")["text"]
graphs = {}


def graph(i):
    if i not in graphs:
        graphs[i] = Graph.from_sentence(texts[i], embedding)
    return graphs[i]


fgw = Fused_Gromov_Wasserstein_distance(alpha=ALPHA, features_metric="sqeuclidean", method="temporal")

## Barycenters

`N_REPEATS` × 3 barycenters per dimension, from the example pools.

In [ ]:
barycenters = {}  # (dimension, repeat) -> [Low, Medium, High]
for dimension in DIMENSIONS:
    pool = pools[dimension]
    for repeat in range(N_REPEATS):
        barycenters[(dimension, repeat)] = [
            compute_barycenter([graph(i) for i in pool.loc[pool["level"] == level, "id"]
                                .sample(M, random_state=SEED + repeat)],
                               alpha=ALPHA, max_iter=BARYCENTER_MAX_ITER)
            for level in LEVELS]

## Classification

Every combination of barycenter dimension (the level predicted) and target set (the dimension manipulated).

In [ ]:
predictions = {}  # (predicted dimension, manipulated dimension) -> DataFrame
for predicted in DIMENSIONS:
    for manipulated in DIMENSIONS:
        start = time.time()
        target = targets[manipulated]
        # distances[t, r, k]: transcript t, repeat r, level k
        distances = np.array([[[fgw.graph_d(graph(i), b) for b in barycenters[(predicted, r)]]
                               for r in range(N_REPEATS)] for i in target["id"]])
        mean_distances = distances.mean(axis=1)
        predictions[(predicted, manipulated)] = pd.DataFrame({
            "id": target["id"].to_numpy(),
            "instructed": target["level"].to_numpy(),
            "predicted": np.array(LEVELS)[mean_distances.argmin(axis=1)],
            "score": mean_distances[:, 0] - mean_distances[:, 2],
            # predicted level of each repeat, for the variability over example draws
            **{f"predicted_{r}": np.array(LEVELS)[distances[:, r].argmin(axis=1)] for r in range(N_REPEATS)},
        })
        print(f"{predicted} barycenters on {manipulated}-manipulated transcripts: "
              f"{len(target)} transcripts, {time.time() - start:.0f} s")

for (predicted, manipulated), pred in predictions.items():
    pred.to_csv(os.path.join(OUTPUT_DIR, f"cross_predictions_{predicted}_on_{manipulated}.csv"), index=False)

## Correlations

Spearman $\rho$ between the predicted level of one dimension and the instructed level of the manipulated dimension (Low = 0, Medium = 1, High = 2), with a 95% bootstrap CI over transcripts. Same with the continuous score. `rho over repeats` is the mean ± SD of $\rho$ when each draw of examples is used alone.

* diagonal (predicted = manipulated): validity, the manipulated dimension is recovered;
* off-diagonal: cross-talk, the level of the **other** dimension shifts with the manipulated one.

In [ ]:
index = {level: k for k, level in enumerate(LEVELS)}
rows = []
for (predicted, manipulated), pred in predictions.items():
    instructed = pred["instructed"].map(index)
    level_corr = spearman_with_ci(pred["predicted"].map(index), instructed)
    score_corr = spearman_with_ci(pred["score"], instructed)
    per_repeat = [spearmanr(pred[f"predicted_{r}"].map(index), instructed)[0] for r in range(N_REPEATS)]
    rows.append({
        "predicted level": predicted, "manipulated (instructed)": manipulated,
        "kind": "validity" if predicted == manipulated else "cross-talk", "n": level_corr["n"],
        "rho (level)": f"{level_corr['rho']:.2f} [{level_corr['rho_ci_low']:.2f}, {level_corr['rho_ci_high']:.2f}]",
        "p (level)": level_corr["p"],
        "rho (score)": f"{score_corr['rho']:.2f} [{score_corr['rho_ci_low']:.2f}, {score_corr['rho_ci_high']:.2f}]",
        "p (score)": score_corr["p"],
        "rho over repeats": f"{np.mean(per_repeat):.2f} ± {np.std(per_repeat, ddof=1):.2f}",
    })

correlations = pd.DataFrame(rows)
correlations.to_csv(os.path.join(OUTPUT_DIR, "cross_correlations.csv"), index=False)
correlations

## Predicted level by instructed level

Share of the transcripts of each instructed level (rows) predicted as each level (columns).

In [ ]:
fig, axes = plt.subplots(len(DIMENSIONS), len(DIMENSIONS), figsize=(10, 9), squeeze=False)
for row, predicted in enumerate(DIMENSIONS):
    for col, manipulated in enumerate(DIMENSIONS):
        pred = predictions[(predicted, manipulated)]
        table = pd.crosstab(pred["instructed"], pred["predicted"], normalize="index").reindex(
            index=LEVELS, columns=LEVELS, fill_value=0)
        sns.heatmap(table, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=axes[row, col])
        axes[row, col].set_xlabel(f"predicted {predicted}")
        axes[row, col].set_ylabel(f"instructed {manipulated}")
        axes[row, col].set_title(f"{predicted} barycenters, {manipulated} manipulated")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "cross_crosstab.png"), dpi=300)
plt.show()

## Score by instructed level

Score $d(\text{Low}) - d(\text{High})$: higher means closer to the High barycenter of the predicted dimension.

In [ ]:
fig, axes = plt.subplots(1, len(DIMENSIONS) ** 2, figsize=(16, 4), sharey=False)
for ax, ((predicted, manipulated), pred) in zip(axes, predictions.items()):
    sns.pointplot(data=pred, x="instructed", y="score", order=LEVELS, errorbar=("ci", 95), ax=ax)
    ax.set_xlabel(f"instructed {manipulated}")
    ax.set_ylabel(f"{predicted} score")
    ax.set_title(f"{predicted} on {manipulated}-manipulated")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "cross_scores.png"), dpi=300)
plt.show()